In [1]:
!pip install -q transformers==5.4.0 sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.2 MB/s eta 0:00:00


Qwen2.5-0.5B: Basic LLM Inference

1. Objective

This notebook performs a basic inference pass through Qwen2.5-0.5B.

The purpose is to understand how an input sentence moves through a
decoder-only Transformer and how we can inspect the model's internal
representations.

The main pipeline is:

Text → Tokenizer → Token IDs → Transformer → Hidden States → Logits



In [12]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


2. Importing the libraries

`import torch`

`from transformers import AutoTokenizer, AutoModelForCausalLM`

PyTorch

torch is used for tensor operations and for running the model on the
GPU or CPU.

Transformers

AutoTokenizer loads the tokenizer associated with the model.

AutoModelForCausalLM loads a causal language model, which predicts the
next token based on the tokens that have already appeared.


3. Selecting the model

`model_name = "Qwen/Qwen2.5-0.5B"`

This specifies the Hugging Face model that we want to use.

Qwen2.5-0.5B is a relatively small causal language model, making it
practical for experimentation on Colab.

In [23]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device)

model.eval() #eval mode

print("Model loaded!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded!


5. Loading the tokenizer

`tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)`

A language model does not directly process raw text. The tokenizer
converts text into token IDs.

Conceptually:

Text
 ↓
Tokenizer
 ↓
Token IDs

For example, our Hindi sentence is broken into tokens, and each token is
represented by an integer ID.

The tokenizer is also responsible for converting token IDs back into
text when we decode generated output.


6. Loading the model

`model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device)`

This downloads and loads the pretrained `Qwen2.5-0.5B model`.

The model is a `decoder-only Transformer`. Its simplified structure
is:

Token IDs<br>
   ↓<br>
Token Embeddings<br>
   ↓<br>
Transformer Layer 1<br>
   ↓<br>
Transformer Layer 2<br>
   ↓<br>
   ...<br>
   ↓<br>
Transformer Layer N<br>
   ↓<br>
Language Model Head<br>
   ↓<br>
Logits<br>

Data type

On a GPU, the script uses bfloat16 to reduce memory usage and make
inference more efficient.

On the CPU, it uses float32.

Moving the model

`.to(device)`

moves the model to the selected CPU/GPU device.

In [14]:
text = "लहरों को शांत देखकर ये मत समझना कि समुद्र में रवानी नहीं है, जब भी उठेंगे, तूफ़ान बन कर उठेंगे।"

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(device)

print("Input IDs:")
print(inputs["input_ids"])

print("\nShape:")
print(inputs["input_ids"].shape)

Input IDs:
tensor([[ 91811,  93948,  44179,  54575,  72314,  47809,  54575,  14925,    114,
          31411,    224,  79238,  14925,     99,  54784,    244,  64704,  44179,
          14925,    107,  34370,  91217,  79238,  68158,  87244, 148971,  60096,
          23868,  47809,  38851,  68158,  87244,  72653, 145256,  85033,  91217,
          54784,    224,  14925,    108, 145535,  31411,    101,  43647,  14925,
            101,  93948,  43647,  72314,  84310,  12619,    230,     11,  14925,
            250, 145799,  14925,    255,  43647,  14925,    231, 146826,  54784,
            224, 145959,  34370,     11,  14925,     97,  12619,    224, 146934,
           5502,    120,  23868,  60096,  14925,    105,  60096,  47809,  44179,
          14925,    231, 146826,  54784,    224, 145959,  34370, 146031]],
       device='cuda:0')

Shape:
torch.Size([1, 89])


In [15]:
with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

11. Running the model

with torch.no_grad():

    outputs = model(

        **inputs,

        output_hidden_states=True

    )

`torch.no_grad()`

We are doing inference, not training.

Therefore, we do not need gradients or the computation graph required
for backpropagation.

Using torch.no_grad() reduces unnecessary memory usage during
inference.

output_hidden_states=True

This is particularly important for our research.

Normally, we mainly care about the final model output. Here, we also
want access to the representations produced at different stages of the
Transformer.

Conceptually:

H₀
 ↓
H₁
 ↓
H₂
 ↓
...
 ↓
Hₗ

where each H represents a hidden representation at a particular stage.

These representations will later allow us to study how much each layer
changes the representation.

In [24]:
print(outputs.logits.shape)
print(len(outputs.hidden_states))


torch.Size([1, 89, 151936])
25


In [25]:
outputs.logits

tensor([[[ 2.5156,  6.9375,  3.8594,  ..., -5.3438, -5.3438, -5.3438],
         [ 7.3438,  5.3438,  5.0312,  ..., -6.8438, -6.8438, -6.8438],
         [ 3.0312,  3.7031,  2.1406,  ..., -8.3125, -8.3125, -8.3125],
         ...,
         [ 8.2500,  7.3750,  2.3594,  ..., -5.1875, -5.1875, -5.1875],
         [12.5625,  9.6875,  4.7812,  ..., -6.2500, -6.2500, -6.2500],
         [ 4.6250,  9.5000,  4.8750,  ..., -9.2500, -9.2500, -9.2500]]],
       device='cuda:0', dtype=torch.bfloat16)

In [18]:
last_hidden = outputs.hidden_states[-1][:, -1, :]

print("Last hidden state:", last_hidden.shape)

Last hidden state: torch.Size([1, 896])


In [22]:
last_logits = outputs.logits[:, -1, :]

print("Last-token logits:", last_logits.shape)

Last-token logits: torch.Size([1, 151936])


In [20]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [21]:
print(inputs["input_ids"].shape)
print(len(outputs.hidden_states))
print(last_hidden.shape)
print(last_logits.shape)

torch.Size([1, 89])
25
torch.Size([1, 896])
torch.Size([1, 151936])


19. Overall inference flow

The complete process in this notebook is:

Hindi Text<br>
    ↓<br>
Tokenizer<br>
    ↓<br>
Token IDs<br>
    ↓<br>
Token Embeddings<br>
    ↓<br>
Transformer Layers<br>
    ↓<br>
Hidden States<br>
    ↓<br>
Final Hidden Representation<br>
    ↓<br>
Language Model Head<br>
    ↓<br>
Logits<br>
    ↓<br>
Next-token prediction<br>